<a href="https://colab.research.google.com/github/jebetancourt1976/AI/blob/main/RAGwithOpenAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG model with OPEN AI as orchestrator

## Upload document and install dependencies

In [ ]:
#" Install this first:" !pip install -q langchain langchain-community langchain-openai faiss-cpu pypdf
#" Install this second:" !pip install -q langchain-text-splitters
!pip install langchain

In [ ]:
#Upload the document (dental policy)
from google.colab import files
uploaded = files.upload()

Saving dental_benefits_summary.pdf to dental_benefits_summary.pdf


## Load and chunk PDF

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
PDF_PATH = "dental_benefits_summary.pdf"

In [ ]:
#Create the loader
loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

print(f"Loaded {len(documents)} pages from PDF")

Loaded 6 pages from PDF


In [ ]:
#Chunk document
splitter=RecursiveCharacterTextSplitter(chunk_size=800,chunk_overlap=100)
chunks=splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")

Split into 36 chunks


## Create the embeddings + FAISS index

In [ ]:
import os
from google.colab import userdata

In [ ]:
#Set the OpenAI key using ColabServices
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
#

In [ ]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
embeddings=OpenAIEmbeddings()

In [ ]:
vectorstore=FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [ ]:
from re import search
#declare the retriever
retriever=vectorstore.as_retriever(search_kwargs={"k": 3})

## Create the Dental Coverage QA chain

In [ ]:
from langchain_openai import ChatOpenAI

In [ ]:

llm=ChatOpenAI(model_name="gpt-4o-mini",
               temperature=0) #We want deterministic answers so Temperature=0)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
prompt = ChatPromptTemplate.from_template("""
Answer the question using ONLY the context below.
If the answer is not in the document, say "Not found in policy.
Use this tool to answer questions about a patient's dental coverage,
including benefits, coverage percentages, exclusions, deductibles,
and annual limits.
Return page numbers as citations.


Context:
{context}

Question:
{question}
""")

In [ ]:
rag_chain = (
    {
        "context": retriever,
        "question": lambda x: x
    }
    | prompt
    | llm
    | StrOutputParser()
)

## Testing of the model

In [ ]:
rag_chain.invoke("what services are not covered?")

'Services not covered include:\n\n1. Services or supplies that are covered in whole or in part under any other part of the Dental Care Plan or under any other plan of group benefits provided by or through your employer (page 2).\n2. Services that Aetna defines as not necessary for the diagnosis, care, or treatment of the condition involved, even if they are prescribed, recommended, or approved by the attending physician or dentist (page 2).\n3. Services intended for treatment of any jaw joint disorder, unless otherwise specified in the Booklet - Certificate (page 2).\n4. Services for space maintainers, except when needed to preserve space resulting from the premature loss of deciduous teeth (page 2).\n5. Services for replacement of a lost, missing, or stolen appliance, and those for replacement of appliances that have been damaged due to abuse, misuse, or neglect (page 2).'